In [2]:
import pointblank as pb
import polars as pl
import re

from pointblank import Validate, load_dataset
from pointblank.segments import Segment

### `seg_matches()`

pb.seg_matches() should work like `matches()`

Examples:
`pb.seg_matches(r"\d-b\w{2}-\d{3}") => Segment(segments=[["1-bcd-345", "5-boe-639", "5-bce-642", "5-bce-642"]])`

#### Define 

`seg_matches` needs access to the actual column values to perform regex matches.

Additional information required:

- Can the column values be injected into the function at runtime?
- Can the function be passed as an expression to be resolved later?

#### Evaluate

- could define new SegMatches class
  - this could have separately defined behaviour in `_seg_expr_from_tuple` and `_apply_segments`

Okay, I think go with SegMatches! Give it the pattern and the case_sensitive parameters. Then the seg_matches() helper function will just call and return SegMatches (same params).

Looking at _seg_expr_from_tuple(), add in the statement to be explicit:

```python
elif isinstance(segment, SegMatches):
            seg_tuples = [(column, segment)]
```

There's quite the chain of evaluations but in _evaluate_segments(), check for a SegMatches object:

```python
if isinstance(segment_value, SegMatches):
    # Resolve the pattern against actual data
    seg_tuples = _seg_expr_from_matches(
        data_tbl=table,
        column=column,
        seg_matches=segment_value
    )
```

And  we need _seg_expr_from_matches() defined and that's going to have a signature like this:

```python
def _seg_expr_from_matches(
    data_tbl: any,
    column: str,
    seg_matches: SegMatches
) -> list[tuple[str, Any]]:
```

^^^ similar to _seg_expr_fromstring() but filters values based on a regex match.

It's really a lot but it follows the same pattern. The laziness of this means that we have to defer evaluation till interrogation time, but I think it's a pattern that can be reused for all future seg helpers!

In [8]:
df = pb.load_dataset()
df.head()

date_time,date,a,b,c,d,e,f
datetime[μs],date,i64,str,i64,f64,bool,str
2016-01-04 11:00:00,2016-01-04,2,"""1-bcd-345""",3,3423.29,true,"""high"""
2016-01-04 00:32:00,2016-01-04,3,"""5-egh-163""",8,9999.99,true,"""low"""
2016-01-05 13:32:00,2016-01-05,6,"""8-kdg-938""",3,2343.23,true,"""high"""
2016-01-06 17:23:00,2016-01-06,2,"""5-jdo-903""",null,3892.4,false,"""mid"""
2016-01-09 12:36:00,2016-01-09,8,"""3-ldm-038""",7,283.94,true,"""low"""


```python
@dataclass
class Matches(ColumnSelector):
    pattern: str
    case_sensitive: bool = False

    def resolve(self, columns: list[str]) -> list[str]:
        matches = (
            [col for col in columns if re.search(self.pattern, col)]
            if self.case_sensitive
            else [col for col in columns if re.search(self.pattern, col, re.IGNORECASE)]
        )
        return matches if matches else []
```

In [10]:
pattern = r"\d-b\w{2}-\d{3}$"
df.filter(pl.col("b").str.contains(pattern))

date_time,date,a,b,c,d,e,f
datetime[μs],date,i64,str,i64,f64,bool,str
2016-01-04 11:00:00,2016-01-04,2,"""1-bcd-345""",3,3423.29,true,"""high"""
2016-01-17 11:27:00,2016-01-17,4,"""5-boe-639""",2,1035.64,false,"""low"""
2016-01-20 04:30:00,2016-01-20,3,"""5-bce-642""",9,837.93,false,"""high"""
2016-01-20 04:30:00,2016-01-20,3,"""5-bce-642""",9,837.93,false,"""high"""


In [43]:
from pointblank.validate import _apply_segments, _seg_expr_from_tuple

In [50]:
seg_expr = _seg_expr_from_tuple(("a", pb.seg_group([1, 2, 3])))[0]
seg_expr

('a', [1, 2, 3])

In [53]:
_apply_segments(df, seg_expr)

date_time,date,a,b,c,d,e,f
datetime[μs],date,i64,str,i64,f64,bool,str
2016-01-04 11:00:00,2016-01-04,2,"""1-bcd-345""",3,3423.29,true,"""high"""
2016-01-04 00:32:00,2016-01-04,3,"""5-egh-163""",8,9999.99,true,"""low"""
2016-01-06 17:23:00,2016-01-06,2,"""5-jdo-903""",null,3892.4,false,"""mid"""
2016-01-20 04:30:00,2016-01-20,3,"""5-bce-642""",9,837.93,false,"""high"""
2016-01-20 04:30:00,2016-01-20,3,"""5-bce-642""",9,837.93,false,"""high"""
2016-01-28 02:51:00,2016-01-28,2,"""7-dmx-010""",8,108.34,false,"""low"""
2016-01-30 11:23:00,2016-01-30,1,"""3-dka-303""",null,2230.09,true,"""high"""
